In [1]:
import os
import numpy as np
import pandas as pd
import mne
import neurokit2 as nk
from scipy.signal import resample
from scipy.signal import welch
import matplotlib.pyplot as plt

In [2]:
F_M_Path = r"F:\data\participants.tsv"
F_M = pd.read_csv(F_M_Path, sep="\t")
F_M

,participant_id,sex
0,sub-001,f
1,sub-002,m
2,sub-003,m
3,sub-004,m
4,sub-005,f
...,...,...
120,sub-121,m
121,sub-122,m
122,sub-123,m
123,sub-124,m


### Function

In [3]:
def Read_ECG(subject_id, base_folder=r"F:\data"):
    subject_folder = os.path.join(base_folder, subject_id)
    ecg_data = {}

    for ses in os.listdir(subject_folder):
        ses_folder = os.path.join(subject_folder, ses)
        if os.path.isdir(ses_folder) and ses.startswith("ses-"):
            ecg_folder = os.path.join(ses_folder, "ecg")
            if os.path.exists(ecg_folder):
                for file in os.listdir(ecg_folder):
                    if file.endswith(".edf"):
                        file_path = os.path.join(ecg_folder, file)
                        raw = mne.io.read_raw_edf(file_path, preload=True)
                        raw.pick_channels(['ECG SD'])
                        raw.rename_channels({'ECG SD': 'ECG'})
                        ecg_data[f"{ses}_{file}"] = raw
                        print(f"Loaded {file} from {ses}")
    
    return ecg_data

In [4]:
def ECG_Dataframe(ecg_dict, subject_id, participants_file):
    participants_df = pd.read_csv(participants_file, sep='\t')
    subject_num = subject_id.replace("sub-", "")
    match = participants_df[
        participants_df['participant_id'].isin([subject_id, subject_num])
    ]

    sex = match['sex'].values[0] if not match.empty else "Unknown"

    data_list = []
    for key, raw in ecg_dict.items():
        ses_file_split = key.split("_")
        ses = ses_file_split[0]

        run = None
        for part in ses_file_split:
            if part.startswith("run-"):
                run = part.replace("run-", "")
                break

        ecg_signal, times = raw.get_data(return_times=True)
        ecg_signal = raw.get_data()[0]

        n_samples = len(ecg_signal)
        sfreq = raw.info['sfreq']

        temp_dict = {
            'subject': subject_num,
            'run': run,
            'sfreq': sfreq,
            'n_samples': n_samples,
            'signal': ecg_signal,
            'times': times,
            'sex': sex
        }
        data_list.append(temp_dict)

    ecg_df = pd.DataFrame(data_list)
    print(f"DataFrame summary created with {len(ecg_df)} rows (files) for subject {subject_id} ({sex}).")
    return ecg_df


In [5]:
def Read_annotation(subject_id, base_folder=r"F:\data"):
    subject_folder = os.path.join(base_folder, subject_id)
    ann_dict = {}

    for ses in os.listdir(subject_folder):
        ses_folder = os.path.join(subject_folder, ses)
        if os.path.isdir(ses_folder) and ses.startswith("ses-"):
            eeg_folder = os.path.join(ses_folder, "eeg")
            if os.path.exists(eeg_folder):
                for file in os.listdir(eeg_folder):
                    if file.endswith(".tsv"):
                        file_path = os.path.join(eeg_folder, file)
                        ann = pd.read_csv(file_path, sep="\t")
                        ann_dict[f"{ses}_{file}"] = ann
                        print(f"Loaded {file} from {ses}")
    return ann_dict

In [6]:
def Annotation_Dataframe(ann_dict, subject_id):
    data_list = []
    subject_num = subject_id.replace("sub-", "")

    for key, ann in ann_dict.items():
        ses_file_split = key.split("_")
        ses = ses_file_split[0]

        run = None
        for part in ses_file_split:
            if part.startswith("run-"):
                run = part.replace("run-", "")
                break

        if all(col in ann.columns for col in ['onset', 'duration', 'eventType']):
            temp_df = pd.DataFrame({
                'subject': subject_num,
                'run': [run] * len(ann),
                'onset': ann['onset'].values,
                'duration': ann['duration'].values,
                'eventType': ann['eventType'].values,
            })
            data_list.append(temp_df)

    df_ann = pd.concat(data_list, ignore_index=True)
    print(f"Total annotations: {len(df_ann)}")
    return df_ann

In [7]:
def merge_edf_events(edf_df, events_df):

    merged_df = pd.merge(
        edf_df,
        events_df,
        on=["subject", "run"],
        how="left"
    )

    return merged_df

In [8]:
def remove_unwanted_events(df):
    df_clean = df[~df["eventType"].str.contains("impd|bckg", case=False, na=False)]
    return df_clean

In [9]:
def preprocess_ecg_df(df):

    for idx, row in df.iterrows():
        signal = row['signal']
        sfreq = row['sfreq']

        clean_signal = nk.ecg_clean(signal, sampling_rate=sfreq, method="neurokit")

        signal_min = np.min(clean_signal)
        signal_max = np.max(clean_signal)
        den = signal_max - signal_min

        if den != 0:
            signal_norm = (clean_signal - signal_min) / den
        else:
            signal_norm = clean_signal

        df.at[idx, 'signal'] = signal_norm

    print("ECG preprocessing complete: bandpass + notch + safe normalize (0-1)")
    return df

In [10]:
def extend_seizure_times(df, pre=30, post=30):

    df = df.copy()
    
    df['onset_extended'] = df['onset'] - pre
    df['onset_extended'] = df['onset_extended'].apply(lambda x: max(x, 0)) 
    
    df['duration_extended'] = df['duration'] + pre + post

    print(f"Seizure times extended: +{pre}s before, +{post}s after.")
    return df

In [11]:
PRE_SEC = 600
POST_SEC = 600

In [12]:
def segment_ecg_windows(df,
                                 window_sec=60,
                                 step_sec=10,
                                 label_threshold=0.8,
                                 use_extended=True):

    segments = []
    window_id = 0

    for (subject, run), g in df.groupby(["subject", "run"], sort=False):
        g = g.reset_index(drop=True)
        sfreq = float(g["sfreq"].iloc[0])

        for idx, row in g.iterrows():
            ecg_signal = row["signal"]
            times = row["times"]

            labels_seq = np.array(["interictal"] * len(ecg_signal))

            if use_extended and 'onset_extended' in row and 'duration_extended' in row:
                ictal_start = row["onset_extended"]
                ictal_end = ictal_start + row["duration_extended"]
            else:
                ictal_start = row.get("onset", None)
                ictal_end = ictal_start + row.get("duration", 0) if ictal_start is not None else None

            if ictal_start is not None and ictal_end is not None:
                for i, t in enumerate(times):
                    if ictal_start <= t < ictal_end:
                        labels_seq[i] = "ictal"
                    elif ictal_start - PRE_SEC <= t < ictal_start:
                        labels_seq[i] = "preictal"
                    elif ictal_end <= t < ictal_end + POST_SEC:
                        labels_seq[i] = "postictal"

            win_len = int(window_sec * sfreq)
            step_len = int(step_sec * sfreq)

            for start_idx in range(0, len(ecg_signal) - win_len + 1, step_len):
                end_idx = start_idx + win_len
                w_sig = ecg_signal[start_idx:end_idx]
                w_times = times[start_idx:end_idx]
                w_labels = labels_seq[start_idx:end_idx]

                unique, counts = np.unique(w_labels, return_counts=True)
                ratio = dict(zip(unique, counts / len(w_labels)))

                if ratio.get("preictal", 0) >= label_threshold:
                    window_label = "preictal"
                elif ratio.get("ictal", 0) >= label_threshold:
                    window_label = "ictal"
                elif ratio.get("postictal", 0) >= label_threshold:
                    window_label = "postictal"
                else:
                    window_label = "normal"

                segments.append({
                    "window_id": window_id,
                    "subject": subject,
                    "run": run,
                    "start_time": w_times[0],
                    "end_time": w_times[-1],
                    "signal": w_sig,
                    "times": w_times,
                    'eventType': row['eventType'],
                    "label": window_label,
                    "preictal_ratio": ratio.get("preictal", 0.0),
                    "ictal_ratio": ratio.get("ictal", 0.0),
                    "postictal_ratio": ratio.get("postictal", 0.0),
                    "window_length": len(w_sig),
                    "sfreq": sfreq,
                    "sex": row.get("sex", "Unknown")
                    
                })
                window_id += 1

    seg_df = pd.DataFrame(segments)
    print(f"Segmentation complete: {len(seg_df)} windows created after extension.")
    return seg_df

### Feature Extraction

#### Detect R_Peaks & RR_Interval & HR_Series

In [13]:
def Detect_R_peaks(signal, times, show_plot=False):
    try:
        signal = np.array(signal, dtype=float)
        times = np.array(times, dtype=float)

        if len(signal) < 2 or len(times) < 2:
            return np.array([]), np.zeros(len(signal)), np.array([]), "window too short"
        if np.any(np.isnan(signal)) or np.any(np.isnan(times)):
            return np.array([]), np.zeros(len(signal)), np.array([]), "NaN detected"

        sfreq = 1 / np.mean(np.diff(times))
        clean_signal = nk.ecg_clean(signal, sampling_rate=sfreq, method="neurokit")
        signals, info = nk.ecg_process(clean_signal, sampling_rate=sfreq)
        R_peaks_idx = info["ECG_R_Peaks"]

        if len(R_peaks_idx) < 2:
            return np.array([]), np.zeros(len(signal)), np.array([]), "less than 2 R-peaks"

        R_peaks_times = times[R_peaks_idx]
        RR_intervals_s = np.diff(R_peaks_times)  # seconds

        # Convert RR_intervals to ms for HRV calculation
        RR_intervals_ms = RR_intervals_s * 1000.0

        HR_series_peaks = 60 / RR_intervals_s  # bpm
        HR_series = np.interp(times[1:], R_peaks_times[1:], HR_series_peaks)
        HR_series = np.insert(HR_series, 0, HR_series[0])

        if show_plot:
            nk.events_plot(R_peaks_idx, signal)

        return R_peaks_times, HR_series, RR_intervals_ms, "success"

    except Exception as e:
        return np.array([]), np.zeros(len(signal)), np.array([]), f"error: {e}"

In [14]:
def Apply_R_peak_Detection(df, show_plot=False):
    results = []
    for i, row in df.iterrows():
        print(f"Processing subject {row['subject']} | run {row['run']} | window {i}")
        signal = row['signal']
        times = row['times']

        R_peaks, HR_series, RR_intervals_ms, status = Detect_R_peaks(signal, times, show_plot=show_plot)

        results.append({
            'subject': row['subject'],
            'run': row['run'],
            'sex': row['sex'],
            'start_time': row['start_time'],
            'end_time': row['end_time'],
            'eventtype': row['eventType'],
            'label': row['label'],
            'signal': signal,
            'times': times,
            'R_peaks': R_peaks if status=="success" else np.array([]),
            'RR_intervals_ms': RR_intervals_ms if status=="success" else np.array([]),
            'HR_series': HR_series if status=="success" else np.zeros(len(signal)),
            'status': status
        })

    df_result = pd.DataFrame(results)
    print(f"Finished R-peak detection for {len(df_result)} windows.")
    return df_result


#### Feature Time Domain General

In [15]:
def Compute_HR_summary(df_hr):
    summary_list = []
    for idx, row in df_hr.iterrows():
        hr_series = np.array(row.get('HR_series', np.zeros_like(row['signal'])))
        if len(hr_series) > 0:
            summary = {
                'meanHR': np.mean(hr_series),
                'stdHR': np.std(hr_series),
                'minHR': np.min(hr_series),
                'maxHR': np.max(hr_series)
            }
        else:
            summary = {'meanHR': np.nan, 'stdHR': np.nan, 'minHR': np.nan, 'maxHR': np.nan}
        summary_list.append(summary)

    df_summary = pd.DataFrame(summary_list)
    df_hr_summary = pd.concat([df_hr.reset_index(drop=True), df_summary], axis=1)
    print(f"HR summary computed for {len(df_hr_summary)} windows.")
    return df_hr_summary

In [16]:
def Compute_Baseline_HR(df_hr, pre_ictal_only=False):
    baseline_dict = {}
    subjects = df_hr['subject'].unique()
    for subj in subjects:
        subj_data = df_hr[df_hr['subject'] == subj]
        if pre_ictal_only:
            subj_clean = subj_data[subj_data['label'] == 'normal']
        else:
            subj_clean = subj_data[~subj_data['label'].isin(['ictal', 'pre-ictal', 'post-ictal'])]
        subj_clean = subj_clean.head(int(600 / 60))  
        baseline_HR = np.nanmean(subj_clean['meanHR'])
        baseline_dict[subj] = baseline_HR
    return baseline_dict

In [17]:
def Compute_Delta_HR(df_hr, baseline_dict):
    delta_list = []
    for _, row in df_hr.iterrows():
        subj = row['subject']
        base = baseline_dict.get(subj, np.nan)
        delta = row['meanHR'] - base if not np.isnan(base) else np.nan
        delta_list.append(delta)
    df_hr['Delta_HR'] = delta_list
    print("ΔHR computed and added to DataFrame.")
    return df_hr

#### Feature HRV

In [18]:
def Compute_HRV(df_rpeaks):
    hrv_list = []
    for idx, row in df_rpeaks.iterrows():
        RR_intervals_ms = row.get('RR_intervals_ms', np.array([]))
        if len(RR_intervals_ms) > 1:
            meanRR = np.mean(RR_intervals_ms)
            SDNN   = np.std(RR_intervals_ms, ddof=1)
            RMSSD  = np.sqrt(np.mean(np.diff(RR_intervals_ms)**2))
            pNN50  = np.sum(np.diff(RR_intervals_ms) > 50.0) / len(RR_intervals_ms) * 100
            meanHRV = 60000.0 / meanRR  # bpm
            HRV_features = {
                'meanRR_ms': meanRR,
                'SDNN_ms': SDNN,
                'RMSSD_ms': RMSSD,
                'pNN50': pNN50,
                'meanHRV_bpm': meanHRV
            }
        else:
            HRV_features = {
                'meanRR_ms': np.nan,
                'SDNN_ms': np.nan,
                'RMSSD_ms': np.nan,
                'pNN50': np.nan,
                'meanHRV_bpm': np.nan
            }
        hrv_list.append(HRV_features)

    df_hrv_features = pd.DataFrame(hrv_list)
    df_hrv = pd.concat([df_rpeaks.reset_index(drop=True), df_hrv_features], axis=1)
    print(f"HRV features computed for {len(df_hrv)} windows using ms standard.")
    return df_hrv

In [19]:
def Add_Missing_ECG_Features(df):
    extra_feats = []
    for idx, row in df.iterrows():
        signal = np.array(row['signal'], dtype=float)
        R_peaks = np.array(row['R_peaks'], dtype=int)
        
        mean_ecg = float(np.mean(signal)) if 'mean_ecg' not in row else row['mean_ecg']
        std_ecg  = float(np.std(signal)) if 'std_ecg' not in row else row['std_ecg']
        rms_ecg  = float(np.sqrt(np.mean(signal**2))) if 'rms_ecg' not in row else row['rms_ecg']
        ptp_ecg  = float(np.ptp(signal)) if 'ptp_ecg' not in row else row['ptp_ecg']
        
        fs = 1 / np.mean(np.diff(row['times'])) if len(row['times']) > 1 else 1.0
        freqs, psd = welch(signal, fs=fs, nperseg=min(1024, len(signal)))
        total_power = float(np.sum(psd)) if 'total_power_ecg' not in row else row['total_power_ecg']
        band_idx = (freqs >= 0.5) & (freqs <= 3)
        bp_0_5_3 = float(np.trapz(psd[band_idx], freqs[band_idx])) if np.any(band_idx) and 'bp_0_5_3_ecg' not in row else row.get('bp_0_5_3_ecg', 0.0)
        
        n_beats = len(R_peaks) if 'n_beats' not in row else row['n_beats']
        
        extra_feats.append({
            'mean_ecg': mean_ecg,
            'std_ecg': std_ecg,
            'rms_ecg': rms_ecg,
            'ptp_ecg': ptp_ecg,
            'total_power_ecg': total_power,
            'bp_0_5_3_ecg': bp_0_5_3,
            'n_beats': n_beats
        })
    
    df_extra = pd.DataFrame(extra_feats)
    df_combined = pd.concat([df.reset_index(drop=True), df_extra], axis=1)
    print(f"Added missing ECG features for {len(df_combined)} windows.")
    return df_combined

#### Apply

In [20]:
def Features(df_cleaned, pre_ictal_only_baseline=False, show_rpeaks=False):
    
    print("\nDetecting R-peaks and computing HR series...")
    df_rpeaks = Apply_R_peak_Detection(df_cleaned, show_plot=show_rpeaks)
    
    print("\nComputing HR summary features...")
    df_hr = Compute_HR_summary(df_rpeaks)
    
    print("\nComputing baseline HR & ΔHR...")
    baseline_dict = Compute_Baseline_HR(df_hr, pre_ictal_only=pre_ictal_only_baseline)
    df_delta_hr = Compute_Delta_HR(df_hr, baseline_dict)
    
    print("\nComputing HRV features (all in ms)...")
    df_hrv = Compute_HRV(df_delta_hr)
    
    print("\nAdding ECG amplitude & frequency features...")
    df_final = Add_Missing_ECG_Features(df_hrv)
    
    print("\nProcessing complete.")
    print("Summary of labels:")
    print(df_final['label'].value_counts())
    
    return {
        "df_rpeaks": df_rpeaks,
        "df_hr": df_hr,
        "baseline_dict": baseline_dict,
        "df_delta_hr": df_delta_hr,
        "df_hrv": df_final
    }

### apply

In [21]:
def Process_ECG_Subject(subject_id, 
                        base_folder=r"F:\data", 
                        participants_file=r"F:\data\participants.tsv",
                        pre=30, post=30,
                        window_sec=60, step_sec=10,
                        label_threshold=0.8,
                        pre_ictal_only_baseline=False,
                        show_rpeaks=False):

    ecg_dict = Read_ECG(subject_id, base_folder=base_folder)
    ecg_df = ECG_Dataframe(ecg_dict, subject_id, participants_file)

    ann_dict = Read_annotation(subject_id, base_folder=base_folder)
    ann_df = Annotation_Dataframe(ann_dict, subject_id)

    merged_df = merge_edf_events(ecg_df, ann_df)

    clean_df = remove_unwanted_events(merged_df)

    preprocessed_df = preprocess_ecg_df(clean_df)

    extended_df = extend_seizure_times(preprocessed_df, pre=pre, post=post)

    seg_df = segment_ecg_windows(extended_df, 
                                 window_sec=window_sec, 
                                 step_sec=step_sec, 
                                 label_threshold=label_threshold,
                                 use_extended=True)

    features_dict = Features(seg_df, 
                             pre_ictal_only_baseline=pre_ictal_only_baseline, 
                             show_rpeaks=show_rpeaks)

    return clean_df, preprocessed_df, features_dict

In [22]:
subject_id = "sub-077"
clean_df, preprocessed_df, features = Process_ECG_Subject(subject_id)

Extracting EDF parameters from F:\data\sub-077\ses-01\ecg\sub-077_ses-01_task-szMonitoring_run-01_ecg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5512191  =      0.000 ... 21531.996 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loaded sub-077_ses-01_task-szMonitoring_run-01_ecg.edf from ses-01
Extracting EDF parameters from F:\data\sub-077\ses-01\ecg\sub-077_ses-01_task-szMonitoring_run-02_ecg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5522687  =      0.000 ... 21572.996 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loaded sub-077_ses-01_task-szMonitoring_run-02_ecg.edf from ses-01
Extracting EDF parameters from F:\data\sub-077\ses-01\ecg\sub-077_ses-01_task-szMonitoring_run-03_ecg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 

c:\Users\Abdelrhamn\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\Abdelrhamn\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


Processing subject 077 | run 10 | window 11066
Processing subject 077 | run 10 | window 11067
Processing subject 077 | run 10 | window 11068
Processing subject 077 | run 10 | window 11069
Processing subject 077 | run 10 | window 11070
Processing subject 077 | run 10 | window 11071
Processing subject 077 | run 10 | window 11072
Processing subject 077 | run 10 | window 11073
Processing subject 077 | run 10 | window 11074
Processing subject 077 | run 10 | window 11075
Processing subject 077 | run 10 | window 11076
Processing subject 077 | run 10 | window 11077
Processing subject 077 | run 10 | window 11078
Processing subject 077 | run 10 | window 11079
Processing subject 077 | run 10 | window 11080
Processing subject 077 | run 10 | window 11081
Processing subject 077 | run 10 | window 11082
Processing subject 077 | run 10 | window 11083
Processing subject 077 | run 10 | window 11084
Processing subject 077 | run 10 | window 11085
Processing subject 077 | run 10 | window 11086
Processing su

C:\Users\Abdelrhamn\AppData\Local\Temp\ipykernel_12576\122617723.py:16: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  bp_0_5_3 = float(np.trapz(psd[band_idx], freqs[band_idx])) if np.any(band_idx) and 'bp_0_5_3_ecg' not in row else row.get('bp_0_5_3_ecg', 0.0)


Added missing ECG features for 11642 windows.

Processing complete.
Summary of labels:
label
normal       10540
preictal       511
postictal      509
ictal           82
Name: count, dtype: int64


In [23]:
df_rpeaks = features['df_rpeaks']
df_hr = features['df_hr']
baseline_dict = features['baseline_dict']
df_delta_hr = features['df_delta_hr']
df_hrv = features['df_hrv']

In [24]:
features_to_show = [
       'meanHR', 'stdHR', 'minHR', 'maxHR', 'Delta_HR', 'meanRR_ms', 'SDNN_ms',
       'RMSSD_ms', 'pNN50', 'meanHRV_bpm', 'mean_ecg', 'std_ecg', 'rms_ecg',
       'ptp_ecg', 'total_power_ecg', 'bp_0_5_3_ecg', 'n_beats'
]

first_row_values =df_hrv.loc[0, features_to_show] 
print(first_row_values)

meanHR              70.605264
stdHR                5.317868
minHR               63.209877
maxHR               90.887574
Delta_HR             0.225398
meanRR_ms          849.581069
SDNN_ms             64.310655
RMSSD_ms            50.482097
pNN50               11.594203
meanHRV_bpm         70.623043
mean_ecg             0.484066
std_ecg              0.058526
rms_ecg              0.487592
ptp_ecg              0.387727
total_power_ecg      0.013564
bp_0_5_3_ecg         0.001487
n_beats                    70
Name: 0, dtype: object


In [25]:
df_hrv.to_csv(f"{subject_id}_ecg_features.csv", index=False)

In [26]:
df_hrv

,subject,run,sex,start_time,end_time,eventtype,label,signal,times,R_peaks,...,RMSSD_ms,pNN50,meanHRV_bpm,mean_ecg,std_ecg,rms_ecg,ptp_ecg,total_power_ecg,bp_0_5_3_ecg,n_beats
0,077,02,m,0.0,59.996094,sz_foc_a_nm,normal,"[0.4841149914818485, 0.4791907714858547, 0.473...","[0.0, 0.00390625, 0.0078125, 0.01171875, 0.015...","[0.625, 1.5390625, 2.453125, 3.33984375, 4.187...",...,50.482097,11.594203,70.623043,0.484066,0.058526,0.487592,0.387727,0.013564,0.001487,70
1,077,02,m,10.0,69.996094,sz_foc_a_nm,normal,"[0.45608021953234323, 0.4549745268257292, 0.45...","[10.0, 10.00390625, 10.0078125, 10.01171875, 1...","[10.3515625, 11.2734375, 12.22265625, 13.15234...",...,51.022369,12.857143,70.528042,0.484434,0.058119,0.487908,0.387727,0.013536,0.001484,71
2,077,02,m,20.0,79.996094,sz_foc_a_nm,normal,"[0.45742603587714953, 0.45937151175697855, 0.4...","[20.0, 20.00390625, 20.0078125, 20.01171875, 2...","[20.7890625, 21.5078125, 22.3671875, 23.199218...",...,44.964314,11.940299,69.003621,0.484495,0.057780,0.487928,0.385284,0.013351,0.001485,68
3,077,02,m,30.0,89.996094,sz_foc_a_nm,normal,"[0.4745134566365701, 0.4721108987792003, 0.470...","[30.0, 30.00390625, 30.0078125, 30.01171875, 3...","[30.62890625, 31.5703125, 32.4921875, 33.33593...",...,40.343577,11.940299,68.144617,0.484602,0.057621,0.488016,0.382236,0.013329,0.001479,68
4,077,02,m,40.0,99.996094,sz_foc_a_nm,normal,"[0.45082680522713586, 0.4500083319469744, 0.44...","[40.0, 40.00390625, 40.0078125, 40.01171875, 4...","[40.3671875, 41.26171875, 42.09375, 42.90625, ...",...,45.160155,11.764706,68.490492,0.484447,0.057548,0.487853,0.382236,0.013264,0.001478,69
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11637,077,10,m,5740.0,5799.996094,sz_foc_a_um,normal,"[0.5997894848205563, 0.5996503936458478, 0.599...","[5740.0, 5740.00390625, 5740.0078125, 5740.011...","[5740.82421875, 5741.41015625, 5742.01171875, ...",...,15.857643,0.000000,103.032477,0.600699,0.005429,0.600723,0.050314,0.000117,0.000004,102
11638,077,10,m,5750.0,5809.996094,sz_foc_a_um,normal,"[0.6063116054042379, 0.6054172928316001, 0.604...","[5750.0, 5750.00390625, 5750.0078125, 5750.011...","[5750.4453125, 5751.0859375, 5751.70703125, 57...",...,16.568257,0.980392,103.148331,0.600699,0.005316,0.600723,0.050314,0.000113,0.000004,103
11639,077,10,m,5760.0,5819.996094,sz_foc_a_um,normal,"[0.5969474824154111, 0.5972765691731954, 0.597...","[5760.0, 5760.00390625, 5760.0078125, 5760.011...","[5760.51953125, 5761.08203125, 5761.6484375, 5...",...,17.684163,0.980392,103.134751,0.600703,0.005158,0.600725,0.050071,0.000106,0.000003,103
11640,077,10,m,5770.0,5829.996094,sz_foc_a_um,normal,"[0.5983176350048912, 0.5981805491688164, 0.598...","[5770.0, 5770.00390625, 5770.0078125, 5770.011...","[5770.87109375, 5771.4609375, 5772.0546875, 57...",...,20.892435,2.000000,102.509343,0.600696,0.005119,0.600718,0.045586,0.000104,0.000003,101


In [27]:
Label_counts = df_hrv['label'].value_counts().reset_index()
Label_counts.columns = ['Label', 'Count']
Label_counts

,Label,Count
0,normal,10540
1,preictal,511
2,postictal,509
3,ictal,82
